# Intra-Session Reproducibility of Instrumental Elbow Extension Velocity in Healthy Subjects

**Anaïs RAGON**  
2026-05-12

---

## 0. Project Accessibility

The project is available on GitHub at this link: [https://github.com/Anais-rgn/Projet_R_Python_M2_REHAB_2026_Anais_Ragon](https://github.com/Anais-rgn/Projet_R_Python_M2_REHAB_2026_Anais_Ragon)

## 1. Scientific Background

### 1.1. Context

Spasticity is a frequent motor symptom following central nervous system injury (stroke, traumatic brain injury, spinal cord injury). It is defined as a **velocity-dependent increase in muscle tone**, causing functional limitations, pain, and significant impairment in daily life activities.

Current clinical assessment relies on standardised scales such as the **Modified Tardieu Scale (MTS)** and the Modified Ashworth Scale (MAS). However, these tools have well-documented limitations, in particular **poor inter- and intra-rater reproducibility**, which limits their reliability for longitudinal monitoring and clinical research.

Inertial Measurement Units (IMU) and portable dynamometers now offer the possibility of obtaining **objective, quantified, and reproducible measurements**. Combining these two technologies could provide a precise instrumental quantification of spasticity.

### 1.2. Overall Study

This pilot study is part of a larger observational study conducted at **CHU Montpellier (Lapeyronie Hospital, MPR department)**:

> *"Evaluation of the reliability of a portable device combining inertial sensors and a dynamometer for the analysis of elbow flexor spasticity in hemiparetic patients."*

**Primary objective:** Assess inter- and intra-rater reproducibility of instrumental spasticity measurement (IMU + dynamometer) in hemiparetic patients, compared to the Modified Tardieu Scale. Target sample: **n = 40 patients**.

### 1.3. This Pilot Study

Before recruiting patients, a **feasibility and reliability pilot study was conducted on healthy subjects** to test the measurement pipeline on clean data and assess the intra-session reproducibility of kinematic parameters and identify technical and methodological limitations before the main study

> **Scientific question:** What is the intra-session reproducibility of instrumental angular velocity during passive elbow extension in healthy subjects?

> **Hypothesis:** Instrumental biomechanical parameters show good intra-session reproducibility in healthy subjects.


## 2. Aim of the Code

The objective of this code is to:
1. Load and parse raw CSV files from the IMU + dynamometer device (K-Push / K-Move)
2. Convert quaternion data into elbow angle time-series (°)
3. Automatically detect extension peaks and flexion troughs in the filtered signal
4. Compute angular velocity per trial (°/s) and categorise trials by speed condition
5. Export a clean dataset for intra-session reproducibility analysis (ICC) in R

## 3. Data Organisation

### 3.1. Experimental Setup

- **3 healthy subjects**, bilateral measurements → **6 limbs** analysed
- **Protocol:** Passive elbow extension at 3 speed conditions × 3 trials, both sides

| Condition | Label | Target duration |
|-----------|-------|-----------------|
| Slow | V1 — `slow` | ~5 s (metronome-guided) |
| Medium | V2 — `medium` | ~2 s |
| Fast | V3 — `fast` | < 1 s |

### 3.2. Instrumentation

| Device | Type | Acquisition frequency |
|--------|------|-----------------------|
| **K-Push** (M122648) | Portable dynamometer | 1000 Hz |
| **K-Move** (S121577) | IMU — wrist sensor | 250 Hz |
| **K-Move** (S121578) | IMU — shoulder sensor | 250 Hz |

### 3.3. Raw Data Files

One CSV file per subject × side (6 files total):

```
data/
├── Data_Ch_D.csv    # Subject Ch — right arm
├── Data_Ch_G.csv    # Subject Ch — left arm
├── Data_Lo_D.csv
├── Data_Lo_G.csv
├── Data_Ca_D.csv
└── Data_Ca_G.csv
```

Each file contains **two merged data blocks**:

| Block | Content | Columns |
|-------|---------|--------|
| **K-Push** | Force data (N) at 1000 Hz | `time (s)` · `CHANNEL_1` |
| **K-Move** | Quaternion data (x, y, z, w) at 250 Hz | `time (s)` · `qx` · `qy` · `qz` · `qw` (× 2 sensors) |

> ⚠️ File structure varied across subjects — a robust parser was required to handle both block order and separator differences.

### 3.4. Derived Variables Analysed

- Mean  **angular velocity** (°/s)

## 4. Notebooks Organisation

The analysis is split across two Python notebooks and one R script:

**`Part1_parsing_angles.ipynb`** (this notebook, Part 1):
- Aim: Load raw CSV files, extract quaternion data, and compute elbow angle time-series
- Input: `Data_[ID]_[D/G].csv` files in `data/` folder
- Output: `all_angles` dictionary — angle time-series per patient × side

**`Part2_detection_analysis.ipynb`** (this notebook, Part 2):
- Aim: Filter the signal, detect movement events, compute angular velocity, and export results
- Input: `all_angles` from Part 1
- Output: `velocity_trials_clean.csv` saved in `data/` folder

**`Part3_ICC_analysis.Rmd`** (R Markdown):
- Aim: Compute ICC(A,1) per speed condition to assess intra-session reproducibility
- Input: `velocity_trials_clean.csv`
- Output: ICC values, p-values, 95% confidence intervals

---
## 5. Code

---

### 5.1. Part 1 — Data Parsing & Angle Computation

#### 5.1.1. Imports and configuration

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from io import StringIO

warnings.filterwarnings('ignore')

#### 5.1.2. Patient configuration

Each patient is identified by their initials. Two CSV files are associated per patient: one for the right arm (D = Droit) and one for the left arm (G = Gauche).

In [ ]:
PATIENTS = {
    "P01": {"id": "Ch", "right": "Data_Ch_D.csv", "left": "Data_Ch_G.csv"},
    "P02": {"id": "Lo", "right": "Data_Lo_D.csv", "left": "Data_Lo_G.csv"},
    "P03": {"id": "Ca", "right": "Data_Ca_D.csv", "left": "Data_Ca_G.csv"},
}

#### 5.1.3. Function definitions

This section defines all functions used in Part 1:

- `load_data`: searches for the CSV file in multiple possible directories and returns its lines
- `process_file`: parses the CSV to extract K-Push (force), K-Move (quaternions), and baseline quaternions
- `_is_float`: utility — checks if a string can be converted to a float
- `quat_conjugate`, `quat_multiply`, `normalize`: quaternion mathematics for 3D rotation computation
- `compute_angle`: converts quaternion time-series to elbow angle (°) using relative rotation from baseline

In [ ]:
def load_data(filename):
    for base_dir in [os.path.join("..", "data"), os.path.join(".", "data"), "."]:
        path = os.path.join(base_dir, filename)
        if os.path.exists(path):
            with open(path, "r", encoding="utf-8") as f:
                return f.readlines()
        if os.path.isdir(base_dir):
            matches = [f for f in os.listdir(base_dir) if f.lower() == filename.lower()]
            if matches:
                with open(os.path.join(base_dir, matches[0]), "r", encoding="utf-8") as f:
                    return f.readlines()
    raise FileNotFoundError(f"File not found: {filename}")


def process_file(filename):
    lines = load_data(filename)
    idx_push = next(i for i, l in enumerate(lines) if "K-Push" in l)
    idx_move = next(i for i, l in enumerate(lines) if "K-Move" in l)

    if idx_push < idx_move:
        push_lines, move_lines = lines[idx_push:idx_move], lines[idx_move:]
    else:
        push_lines, move_lines = lines[idx_push:], lines[idx_move:idx_push]

    # --- K-Push ---
    header_push = next(i for i, l in enumerate(push_lines) if "temps" in l)
    df_push = pd.read_csv(
        StringIO("".join(push_lines[header_push:])),
        sep=r"\t|,", engine="python"
    ).dropna(axis=1, how='all')
    df_push = df_push.rename(columns={"temps (seconde)": "time", "CHANNEL_1": "force"})
    df_push = df_push[["time", "force"]].apply(pd.to_numeric, errors='coerce').dropna()

    # --- Baseline quaternions ---
    baseline = {"wrist": [], "shoulder": []}
    for line in move_lines:
        if "Quaternion de base" in line:
            parts = line.strip().split("\t") if "\t" in line else line.strip().split(",")
            values = [float(x) for x in parts[1:] if _is_float(x)][:4]
            if "S121577" in parts[0]:
                baseline["wrist"] = values
            elif "S121578" in parts[0]:
                baseline["shoulder"] = values

    # --- K-Move ---
    header_move = next(i for i, l in enumerate(move_lines) if "temps" in l)
    rows = [
        (l.strip().split("\t") if "\t" in l else l.strip().split(","))
        for l in move_lines[header_move + 1:]
    ]
    df_move = pd.DataFrame(rows).dropna(axis=1, how='all')
    df_move = df_move.apply(pd.to_numeric, errors='coerce').ffill().bfill()

    df_wrist    = df_move.iloc[:, [0,1,2,3,4]].copy()
    df_shoulder = df_move.iloc[:, [0,6,7,8,9]].copy()
    for df in [df_wrist, df_shoulder]:
        df.columns = ["time", "qx", "qy", "qz", "qw"]

    return df_push, df_wrist, df_shoulder, baseline


def _is_float(s):
    try:
        float(s)
        return True
    except ValueError:
        return False


def quat_conjugate(q):
    return np.array([-q[0], -q[1], -q[2], q[3]])


def quat_multiply(q1, q2):
    x1, y1, z1, w1 = q1
    x2, y2, z2, w2 = q2
    return np.array([
        w1*x2 + x1*w2 + y1*z2 - z1*y2,
        w1*y2 - x1*z2 + y1*w2 + z1*x2,
        w1*z2 + x1*y2 - y1*x2 + z1*w2,
        w1*w2 - x1*x2 - y1*y2 - z1*z2
    ])


def normalize(q):
    return q / np.linalg.norm(q)


def compute_angle(df_quat, baseline_quat, baseline_deg=30):
    q_base_inv = quat_conjugate(normalize(np.array(baseline_quat)))
    angles = []
    for i in range(len(df_quat)):
        q = normalize(np.array([
            df_quat["qx"].iloc[i], df_quat["qy"].iloc[i],
            df_quat["qz"].iloc[i], df_quat["qw"].iloc[i]
        ]))
        q_rel = quat_multiply(q_base_inv, q)
        qx, qy, qz, qw = q_rel
        angle = np.arctan2(2*(qw*qy + qx*qz), 1 - 2*(qy**2 + qz**2))
        angles.append(angle)
    angles = np.degrees(np.unwrap(angles))
    angles -= angles[0]
    if np.mean(angles) < 0:
        angles = -angles
    angles += baseline_deg
    return df_quat[["time"]].assign(angle=angles)

#### 5.1.4. Main loop — load, parse and compute angles for all patients

For each patient and each side, the file is parsed and the elbow angle time-series is computed. Results are stored in `all_angles[patient][side]` as a DataFrame with columns `time` and `angle` (°).

In [ ]:
all_angles = {}

for p_name, p_info in PATIENTS.items():
    print(f"\n{'='*40}\nPatient: {p_name} ({p_info['id']})\n{'='*40}")
    all_angles[p_name] = {}

    for side in ["right", "left"]:
        try:
            df_push, df_wrist, df_shoulder, baseline = process_file(p_info[side])
        except FileNotFoundError as e:
            print(f"[WARNING] {e}")
            continue

        if len(baseline["wrist"]) != 4:
            print(f"[WARNING] Missing baseline for {p_name}/{side}")
            continue

        df_angles = compute_angle(df_wrist, baseline["wrist"])
        all_angles[p_name][side] = df_angles

        print(f"{side}: {df_angles['angle'].min():.1f}° → {df_angles['angle'].max():.1f}°")

        plt.figure(figsize=(10, 4))
        plt.plot(df_angles["time"], df_angles["angle"])
        plt.title(f"{p_name} ({p_info['id']}) — {side} | Elbow angle over time")
        plt.xlabel("Time (s)")
        plt.ylabel("Angle (°)")
        plt.grid()
        plt.tight_layout()
        plt.show()

---
### 5.2. Part 2 — Event Detection & Velocity Analysis

#### 5.2.1. Parameters

Global parameters controlling the detection pipeline:

In [ ]:
from scipy.signal import butter, filtfilt, find_peaks

N_MAX         = 9    # max number of extension peaks detected
N_MIN         = 10   # max number of flexion troughs detected
MIN_PROM      = 15   # minimum peak prominence (degrees)
MIN_DIST_SEC  = 1.5  # minimum distance between two peaks (seconds)
FILTER_CUTOFF = 10   # low-pass filter cutoff frequency (Hz)

#### 5.2.2. Function definitions

- `filtrer_signal`: applies a 4th-order zero-phase Butterworth low-pass filter to remove high-frequency noise
- `detecter_evenements`: detects extension peaks and flexion troughs in the filtered angle signal
- `afficher_et_stocker`: stores results and displays the detection plot

> ⚠️ **Known limitation:** Trough (flexion minimum) detection is unreliable. Despite multiple strategies tested (inverted find_peaks, segment-based search, prominence thresholds, np.pad, ffill/bfill), troughs are sometimes detected during rest periods rather than actual flexion. This leads to incorrect angular velocity values for some trials.

In [ ]:
def filtrer_signal(signal, cutoff, fs):
    b, a = butter(4, min(cutoff / (0.5 * fs), 0.99), btype='low')
    return filtfilt(b, a, signal)


def detecter_evenements(df_angles):
    time   = df_angles["time"].values
    angle  = df_angles["angle"].values
    fs     = 1 / np.mean(np.diff(time))
    signal_f = filtrer_signal(angle, FILTER_CUTOFF, fs)
    dist   = int(fs * MIN_DIST_SEC)

    # Extension peaks
    pics, _ = find_peaks(signal_f, distance=dist, prominence=MIN_PROM)
    if len(pics) > N_MAX:
        pics = np.sort(pics[np.argsort(signal_f[pics])[::-1][:N_MAX]])

    # Flexion troughs (one before each peak)
    creux = []
    for i in range(len(pics)):
        debut = int(max(0, pics[i] - 3.5 * fs)) if i == 0 else pics[i - 1]
        fin   = pics[i]
        seg   = signal_f[debut:fin]
        if len(seg) > 5:
            creux.append(debut + np.argmin(seg))

    # Trough after last peak
    seg = signal_f[pics[-1]:]
    if len(pics) > 0 and len(seg) > 5:
        creux.append(pics[-1] + np.argmin(seg))

    creux = np.unique(creux)[:N_MIN]
    if len(creux) < N_MIN:
        creux = np.pad(creux, (0, N_MIN - len(creux)), mode='edge')

    return np.array(creux, dtype=int), pics, signal_f


def afficher_et_stocker(df_angles, creux, pics, signal_f, patient, cote, resultats):
    resultats.setdefault(patient, {})[cote] = {
        "angles": df_angles, "signal_filtre": signal_f,
        "flexions": creux, "extensions": pics
    }
    time = df_angles["time"].values
    plt.figure(figsize=(14, 5))
    plt.plot(time, signal_f, label="Filtered signal", color="steelblue")
    if len(creux) > 0:
        plt.plot(time[creux], signal_f[creux], "go", ms=8, label="Flexion (trough)")
    if len(pics) > 0:
        plt.plot(time[pics], signal_f[pics], "ro", ms=8, label="Extension (peak)")
    plt.title(f"{patient} — {cote} | Event detection")
    plt.xlabel("Time (s)"), plt.ylabel("Angle (°)")
    plt.legend(), plt.grid(), plt.tight_layout(), plt.show()

#### 5.2.3. Main detection loop

In [ ]:
resultats = {}
for patient, pdata in all_angles.items():
    for cote, df_angles in pdata.items():
        print(f"\n=== DETECTION: {patient} | {cote.upper()} ===")
        creux, pics, signal_f = detecter_evenements(df_angles)
        afficher_et_stocker(df_angles, creux, pics, signal_f, patient, cote, resultats)

#### 5.2.4. Velocity computation and categorisation by speed

For each movement event, angular velocity is computed as:

$$\text{velocity} = \frac{\text{amplitude}}{\text{duration}} = \frac{\theta_{max} - \theta_{min}}{t_{end} - t_{start}} \quad (°/s)$$

Trials are labelled by speed category based on their order: trials 1–3 = **slow**, 4–6 = **medium**, 7–9 = **fast**.

In [ ]:
COULEURS   = {'slow': 'blue', 'medium': 'orange', 'fast': 'red', 'extra': 'gray'}
CATEGORIES = ['slow'] * 3 + ['medium'] * 3 + ['fast'] * 3

resume = {}
all_events_long = []

for patient, pdata in resultats.items():
    resume[patient] = {}
    for cote, data in pdata.items():
        print(f"\n===== EVENTS: {patient} | {cote.upper()} =====")
        signal = data["angles"]["angle"].values
        time   = data["angles"]["time"].values
        creux  = data["flexions"]
        pics   = data["extensions"]

        if len(creux) < 2 or len(pics) == 0:
            print("⚠️ Not enough peaks")
            continue

        events = []
        for i in range(min(len(pics), len(creux) - 1)):
            t0, t1   = time[creux[i]], time[pics[i]]
            duree    = t1 - t0
            amplitude = signal[pics[i]] - signal[creux[i]]
            events.append({
                'event_id': i + 1,
                'min_idx': creux[i], 'max_idx': pics[i],
                'speed_category': CATEGORIES[i] if i < len(CATEGORIES) else 'extra',
                'velocity': amplitude / duree if duree > 0 else 0,
                't_start': t0, 't_end': t1
            })
            all_events_long.append({
                'subject': patient, 'side': cote,
                'trial': i + 1,
                'speed': events[-1]['speed_category'],
                'velocity': events[-1]['velocity']
            })

        df_ev = pd.DataFrame(events)
        resume[patient][cote] = df_ev
        print(df_ev[['event_id', 'speed_category', 'velocity']].to_string(index=False))

        # Coloured plot
        plt.figure(figsize=(14, 5))
        plt.plot(time, signal, 'k-', linewidth=0.8, alpha=0.4)
        for ev in events:
            c   = COULEURS[ev['speed_category']]
            idx = np.arange(ev['min_idx'], ev['max_idx'] + 1)
            plt.plot(time[idx], signal[idx], color=c, linewidth=2.5)
            plt.plot(time[ev['min_idx']], signal[ev['min_idx']], 'o', color=c)
            plt.plot(time[ev['max_idx']], signal[ev['max_idx']], 's', color=c)
        plt.title(f"{patient} — {cote} | Events by speed (blue=slow · orange=medium · red=fast)")
        plt.xlabel("Time (s)"), plt.ylabel("Angle (°)")
        plt.grid(True), plt.tight_layout(), plt.show()

        print("\nSTATISTICS:")
        for cat in ['slow', 'medium', 'fast']:
            subset = df_ev[df_ev['speed_category'] == cat]
            if len(subset) > 0:
                print(f"  {cat.upper()} (n={len(subset)}) — mean velocity = {subset['velocity'].mean():.1f} °/s")

---
### 5.3. Part 3 — Data Export for ICC Analysis

This section standardises column names, numbers each trial within its group, and exports the final dataset to CSV for analysis in R.

In [ ]:
# Build df_full from all_events_long
df_full = pd.DataFrame(all_events_long)

# Standardise column names
df_full = df_full.rename(columns={
    "subject": "patient_id",
    "speed_category": "speed"
})

# Harmonise speed labels
df_full["speed"] = df_full["speed"].replace({
    "lente": "slow", "moyenne": "medium", "rapide": "fast"
})

# Number each trial within group (patient × side × speed)
df_full = df_full.sort_values(by=["patient_id", "side", "speed"])
df_full["trial"] = (
    df_full
        .groupby(["patient_id", "side", "speed"])
        .cumcount() + 1
)

# Export
df_full.to_csv("data/velocity_trials_clean.csv", index=False)
print("✅ CSV ready for ICC analysis")
print(f"Shape: {df_full.shape[0]} rows × {df_full.shape[1]} columns")
print("\nTable 1. Preview — velocity_trials_clean.csv")
df_full.head(9)

---
## 7. R Analysis — Intra-Session Reproducibility (ICC)

### 7.1. Objective
The angular velocity values exported from Part 3 (`velocity_trials_clean.csv`)
are analysed in R to assess **intra-session reproducibility** using the
Intraclass Correlation Coefficient (ICC).

### 7.2. Method
ICC is computed using the `irr` package with the following model:

| Parameter | Value | Meaning |
|-----------|-------|---------|
| `model` | `"twoway"` | Subjects and trials both treated as random effects |
| `type` | `"agreement"` | Absolute agreement — penalises systematic differences |
| `unit` | `"single"` | Reliability of a single trial → ICC(A,1) |

Results are interpreted using **Koo & Mae (2016)** thresholds:
poor < 0.50 / moderate 0.50–0.75 / good 0.75–0.90 / excellent > 0.90

### 7.3. Key Results

| Speed | ICC(A,1) | p-value | 95% CI | Interpretation |
|-------|----------|---------|--------|----------------|
| Slow | 0.27 | 0.067 | [–0.06 ; 0.78] | Poor |
| Medium | 0.11 | 0.301 | [–0.28 ; 0.73] | Poor |
| Fast | 0.77 | < 0.001 | [0.38 ; 0.96] | Good |

### 7.4. Critical Interpretation
These results must be interpreted with **major caution** for two cumulative reasons:

1. **Unreliable trough detection** → several velocity values are incorrect
2. **Insufficient sample size (n = 6 limbs)** → very wide confidence intervals

The good ICC for the fast condition (0.77) may reflect the more stereotyped
nature of ballistic fast movements, which are less sensitive to trough
misdetection due to their short duration.

> These ICC values do not carry valid clinical meaning.
> They provide methodological insight for improving the pipeline
> before the main clinical study.

### 7.3. Conclusion

This pilot study validates the **end-to-end pipeline** (raw file → angle → velocity → ICC) and identifies the key technical challenge to resolve before the main clinical study: reliable automatic detection of movement onset (flexion minimum).

---
## 8. References

Koo TK, Mae AY. A Guideline of Selecting and Reporting Intraclass Correlation Coefficients for Reliability Research. *J Chiropr Med.* 2016;15(2):155–163.

Leuenberger K, Gonzenbach R, Wachter S, et al. A method to qualitatively assess arm use in stroke survivors at home. *Med Biol Eng Comput.* 2017;55:141–150.

Shrout PE, Fleiss JL. Intraclass correlations: uses in assessing rater reliability. *Psychol Bull.* 1979;86(2):420–428.